# Data Wrangling
## Lab 4 — Real Estate Market

**Dataset:** California Housing dataset from `sklearn.datasets`

### Aim
To perform simple data wrangling on housing-price data.

### Tasks
1. Load the dataset.
2. Clean column names.
3. Handle missing values.
4. Merge an additional small dataset.
5. Filter the data.
6. Encode categorical data.
7. Calculate summary statistics.
8. Handle simple outliers.


In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing

pd.set_option("display.max_columns", None)


## 1. Load the dataset

The **California Housing dataset** is available through `sklearn.datasets`.


In [2]:
# Load the California Housing dataset
housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Clean the column names

In [3]:
# Make column names easier to use
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

# Rename the target column
df = df.rename(columns={"medhouseval": "house_price"})

# Actual sklearn column names are kept in lowercase (e.g. averooms, medinc).

print("Clean column names:")
print(df.columns.tolist())


Clean column names:
['medinc', 'houseage', 'averooms', 'avebedrms', 'population', 'aveoccup', 'latitude', 'longitude', 'house_price']


## 3. Check and handle missing values

In [4]:
print("Missing values before cleaning:")
print(df.isnull().sum())

# Fill missing numerical values with the median
df = df.fillna(df.median(numeric_only=True))

print("\nMissing values after cleaning:")
print(df.isnull().sum())


Missing values before cleaning:
medinc         0
houseage       0
averooms       0
avebedrms      0
population     0
aveoccup       0
latitude       0
longitude      0
house_price    0
dtype: int64

Missing values after cleaning:
medinc         0
houseage       0
averooms       0
avebedrms      0
population     0
aveoccup       0
latitude       0
longitude      0
house_price    0
dtype: int64


## 4. Add a simple categorical column

The original dataset mainly contains numerical information. We create a simple **property type** column so that categorical-data handling can be demonstrated.


In [5]:
# Create a simple property type
# If a house has more than 5 average rooms, call it "Large".
df["property_type"] = np.where(
    df["averooms"] > 5,
    "Large",
    "Small"
)

display(df[["averooms", "property_type"]].head())


,averooms,property_type
0,6.984127,Large
1,6.238137,Large
2,8.288136,Large
3,5.817352,Large
4,6.281853,Large


## 5. Merge additional information

In [6]:
# Create a small additional dataset based on the income level
# This represents simple neighborhood information.
neighborhood_info = pd.DataFrame({
    "income_group": ["Low", "Medium", "High"],
    "amenity_score": [4, 7, 9]
})

# Create the same income group in the housing data
df["income_group"] = pd.cut(
    df["medinc"],
    bins=[0, 2.5, 5, np.inf],
    labels=["Low", "Medium", "High"]
)

# Merge the two datasets
df = df.merge(
    neighborhood_info,
    on="income_group",
    how="left"
)

display(df.head())


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price,property_type,income_group,amenity_score
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,Large,High,9
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,Large,High,9
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,Large,High,9
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,Large,High,9
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,Large,Medium,7


## 6. Filter the data

In [7]:
# Select houses with:
# - more than 5 rooms
# - house price greater than 2
filtered_df = df[
    (df["averooms"] > 5) &
    (df["house_price"] > 2)
]

print("Filtered data:")
display(filtered_df.head())

print("Number of filtered records:", len(filtered_df))


Filtered data:


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price,property_type,income_group,amenity_score
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,Large,High,9
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,Large,High,9
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,Large,High,9
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,Large,High,9
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,Large,Medium,7


Number of filtered records: 5978


## 7. Encode categorical variables

In [8]:
# One-hot encode the property_type column
df_encoded = pd.get_dummies(
    df,
    columns=["property_type"],
    drop_first=True
)

display(df_encoded.head())


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price,income_group,amenity_score,property_type_Small
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,High,9,False
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,High,9,False
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,High,9,False
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,High,9,False
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,Medium,7,False


## 8. Calculate summary statistics

In [9]:
# Average house price by property type
average_price = (
    df.groupby("property_type")["house_price"]
    .mean()
)

print("Average house price by property type:")
print(average_price)

print("\nOverall summary:")
display(
    df[["medinc", "averooms", "house_price"]].describe()
)


Average house price by property type:
property_type
Large    2.264487
Small    1.801260
Name: house_price, dtype: float64

Overall summary:


,medinc,averooms,house_price
count,20640.000000,20640.000000,20640.000000
mean,3.870671,5.429000,2.068558
std,1.899822,2.474173,1.153956
min,0.499900,0.846154,0.149990
25%,2.563400,4.440716,1.196000
50%,3.534800,5.229129,1.797000
75%,4.743250,6.052381,2.647250
max,15.000100,141.909091,5.000010


## 9. Handle outliers

In [10]:
# Simple outlier handling using the IQR method
Q1 = df["house_price"].quantile(0.25)
Q3 = df["house_price"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Remove values outside the limits
clean_df = df[
    (df["house_price"] >= lower_limit) &
    (df["house_price"] <= upper_limit)
].copy()

print("Rows before removing outliers:", len(df))
print("Rows after removing outliers :", len(clean_df))


Rows before removing outliers: 20640
Rows after removing outliers : 19569


## 10. Final cleaned dataset

In [11]:
print("Final dataset shape:", clean_df.shape)
display(clean_df.head())

# Save the cleaned dataset
clean_df.to_csv("Cleaned_RealEstate_Prices.csv", index=False)

print("Cleaned dataset exported successfully.")


Final dataset shape: (19569, 12)


,medinc,houseage,averooms,avebedrms,population,aveoccup,latitude,longitude,house_price,property_type,income_group,amenity_score
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,Large,High,9
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,Large,High,9
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,Large,High,9
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,Large,High,9
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,Large,Medium,7


Cleaned dataset exported successfully.


## Conclusion

The real estate data was wrangled by:
- Loading the California Housing dataset from sklearn
- Cleaning column names
- Handling missing values
- Adding and merging neighborhood information
- Filtering records
- Encoding a categorical variable
- Calculating average house prices
- Detecting and removing simple outliers
- Exporting the cleaned dataset
